# Food Recommendation System
## Dataset Analysis

### Cilj analize

Cilj ove analize je da detaljno ispitamo dostupne podatke o receptima i interakcijama korisnika sa receptima. 
Analiza će obuhvatiti strukturu podataka, broj recepata i korisnika, distribuciju ocena, nedostajuće vrednosti, 
duplikate i druge karakteristike koje su važne za kasniju izgradnju sistema preporuke.

U ovoj fazi ne treniramo modele. Cilj je da razumemo dataset i na osnovu njegovih karakteristika 
odredimo kako ćemo pripremiti podatke za modele preporuke.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
recipes_path = "../../datasets/RAW_recipes.csv"

recipes = pd.read_csv(recipes_path)

print("Recipes dataset successfully loaded.")

In [ ]:
interactions_path = "../../datasets/RAW_interactions.csv"

interactions = pd.read_csv(interactions_path)

print("Interactions dataset successfully loaded.")

In [ ]:
recipes.head()

In [ ]:
interactions.head()

In [ ]:
print("Recipes shape:", recipes.shape)
print("Interactions shape:", interactions.shape)

In [ ]:
print("Number of recipes:", recipes["id"].nunique())
print("Number of users:", interactions["user_id"].nunique())
print("Number of ratings:", len(interactions))

In [ ]:
recipes.info()

In [ ]:
interactions.info()

In [ ]:
recipes_missing = recipes.isnull().sum()

print("Missing values in recipes:")
print(recipes_missing)

In [ ]:
interactions_missing = interactions.isnull().sum()

print("Missing values in interactions:")
print(interactions_missing)

In [ ]:
recipes_missing_summary = pd.DataFrame({
    "missing_count": recipes.isnull().sum(),
    "missing_percentage": recipes.isnull().mean() * 100
})

recipes_missing_summary

In [ ]:
interactions_missing_summary = pd.DataFrame({
    "missing_count": interactions.isnull().sum(),
    "missing_percentage": interactions.isnull().mean() * 100
})

interactions_missing_summary

In [ ]:
print("Duplicate rows in recipes:", recipes.duplicated().sum())
print("Duplicate rows in interactions:", interactions.duplicated().sum())

In [ ]:
user_recipe_counts = (
    interactions
    .groupby(["user_id", "recipe_id"])
    .size()
    .reset_index(name="interaction_count")
)

print(
    "Number of user-recipe pairs with more than one interaction:",
    (user_recipe_counts["interaction_count"] > 1).sum()
)

In [ ]:
print(
    "Maximum interactions for a single user-recipe pair:",
    user_recipe_counts["interaction_count"].max()
)   

In [ ]:
rating_counts = interactions["rating"].value_counts().sort_index()

rating_counts

In [ ]:
plt.figure(figsize=(8, 5))

rating_counts.plot(kind="bar")

plt.title("Distribution of Recipe Ratings")
plt.xlabel("Rating")
plt.ylabel("Number of Interactions")
plt.xticks(rotation=0)

plt.show()

In [ ]:
rating_percentages = (
        interactions["rating"]
        .value_counts(normalize=True)
        .sort_index()
        * 100
)

rating_percentages

In [ ]:
num_users = interactions["user_id"].nunique()
num_recipes = recipes["id"].nunique()
num_interactions = len(interactions)

total_possible_interactions = num_users * num_recipes

sparsity = 1 - (num_interactions / total_possible_interactions)

print("Users:", num_users)
print("Recipes:", num_recipes)
print("Observed interactions:", num_interactions)
print("Possible interactions:", total_possible_interactions)
print("Sparsity:", sparsity)
print("Sparsity (%):", sparsity * 100)

In [ ]:
user_activity = interactions.groupby("user_id").size()

print("Average interactions per user:", user_activity.mean())
print("Median interactions per user:", user_activity.median())
print("Minimum interactions per user:", user_activity.min())
print("Maximum interactions per user:", user_activity.max())

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 5))

plt.hist(user_activity, bins=50)

plt.title("Distribution of User Activity")
plt.xlabel("Number of Interactions per User")
plt.ylabel("Number of Users")

plt.show()

In [ ]:
user_activity_50 = user_activity[user_activity <= 50]

plt.figure(figsize=(10, 5))

plt.hist(
    user_activity_50,
    bins=50,
    range=(0.5, 50.5)
)

plt.title("Distribution of User Activity (1-50 interactions)")
plt.xlabel("Number of Interactions per User")
plt.ylabel("Number of Users")

plt.xticks(range(1, 51, 5))

plt.show()

In [ ]:
recipe_popularity = interactions.groupby("recipe_id").size()

print("Average interactions per recipe:", recipe_popularity.mean())
print("Median interactions per recipe:", recipe_popularity.median())
print("Minimum interactions per recipe:", recipe_popularity.min())
print("Maximum interactions per recipe:", recipe_popularity.max())

In [ ]:
recipe_popularity_50 = recipe_popularity[recipe_popularity <= 50]

plt.figure(figsize=(10, 5))

plt.hist(
    recipe_popularity_50,
    bins=50,
    range=(0.5, 50.5)
)

plt.title("Distribution of Recipe Popularity (1-50 interactions)")
plt.xlabel("Number of Interactions per Recipe")
plt.ylabel("Number of Recipes")

plt.xticks(range(1, 51, 5))

plt.show()

In [ ]:
top_recipes = (
    recipe_popularity
    .sort_values(ascending=False)
    .head(10)
)

top_recipes

In [ ]:
top_recipes_info = (
    top_recipes
    .rename("interaction_count")
    .reset_index()
    .merge(
        recipes[["id", "name"]],
        left_on="recipe_id",
        right_on="id",
        how="left"
    )
    [["recipe_id", "name", "interaction_count"]]
)

top_recipes_info

In [ ]:
explicit_ratings = interactions[interactions["rating"] > 0].copy()

print("Original interactions:", len(interactions))
print("Explicit ratings:", len(explicit_ratings))
print("Removed rating=0:", len(interactions) - len(explicit_ratings))

In [ ]:
user_counts = explicit_ratings.groupby("user_id").size()

thresholds = [1, 2, 3, 5, 10]

for threshold in thresholds:
    users_remaining = (user_counts >= threshold).sum()
    interactions_remaining = user_counts[user_counts >= threshold].sum()

    print(
        f"Minimum {threshold}: "
        f"{users_remaining} users, "
        f"{interactions_remaining} interactions"
    )

In [ ]:
recipe_counts = explicit_ratings.groupby("recipe_id").size()

thresholds = [1, 2, 3, 5, 10]

for threshold in thresholds:
    recipes_remaining = (recipe_counts >= threshold).sum()
    interactions_remaining = recipe_counts[recipe_counts >= threshold].sum()

    print(
        f"Minimum {threshold}: "
        f"{recipes_remaining} recipes, "
        f"{interactions_remaining} interactions"
    )
    

In [ ]:
MIN_USER_RATINGS = 5
MIN_RECIPE_RATINGS = 5

filtered_ratings = explicit_ratings.copy()

while True:
    user_counts = filtered_ratings.groupby("user_id").size()
    recipe_counts = filtered_ratings.groupby("recipe_id").size()

    valid_users = user_counts[user_counts >= MIN_USER_RATINGS].index
    valid_recipes = recipe_counts[recipe_counts >= MIN_RECIPE_RATINGS].index

    new_filtered_ratings = filtered_ratings[
        filtered_ratings["user_id"].isin(valid_users)
        & filtered_ratings["recipe_id"].isin(valid_recipes)
        ].copy()

    if len(new_filtered_ratings) == len(filtered_ratings):
        break

    filtered_ratings = new_filtered_ratings

print("Filtered interactions:", len(filtered_ratings))
print("Filtered users:", filtered_ratings["user_id"].nunique())
print("Filtered recipes:", filtered_ratings["recipe_id"].nunique())

In [ ]:
from sklearn.model_selection import train_test_split

train_ratings, test_ratings = train_test_split(
    filtered_ratings,
    test_size=0.20,
    random_state=42
)

print("Train interactions:", len(train_ratings))
print("Test interactions:", len(test_ratings))
print("Total interactions:", len(train_ratings) + len(test_ratings))

In [ ]:
train_users = set(train_ratings["user_id"])
test_users = set(test_ratings["user_id"])

train_recipes = set(train_ratings["recipe_id"])
test_recipes = set(test_ratings["recipe_id"])

cold_start_users = test_users - train_users
cold_start_recipes = test_recipes - train_recipes

print("Cold-start users in test:", len(cold_start_users))
print("Cold-start recipes in test:", len(cold_start_recipes))

In [ ]:
from sklearn.model_selection import train_test_split

train_ratings, test_ratings = train_test_split(
    filtered_ratings,
    test_size=0.20,
    random_state=42
)

# Provera cold-start korisnika i recepata
while True:
    train_users = set(train_ratings["user_id"])
    train_recipes = set(train_ratings["recipe_id"])

    test_users = set(test_ratings["user_id"])
    test_recipes = set(test_ratings["recipe_id"])

    cold_start_users = test_users - train_users
    cold_start_recipes = test_recipes - train_recipes

    if len(cold_start_users) == 0 and len(cold_start_recipes) == 0:
        break

    cold_start_mask = (
            test_ratings["user_id"].isin(cold_start_users)
            | test_ratings["recipe_id"].isin(cold_start_recipes)
    )

    train_ratings = pd.concat(
        [train_ratings, test_ratings[cold_start_mask]],
        ignore_index=True
    )

    test_ratings = test_ratings[~cold_start_mask].copy()

print("Train interactions:", len(train_ratings))
print("Test interactions:", len(test_ratings))

print("Cold-start users:", len(cold_start_users))
print("Cold-start recipes:", len(cold_start_recipes))

In [ ]:
import os

os.makedirs("../datasets/processed", exist_ok=True)

print("Processed folder is ready.")

In [ ]:
filtered_ratings.to_csv(
    "../datasets/processed/filtered_ratings.csv",
    index=False
)

train_ratings.to_csv(
    "../datasets/processed/train_ratings.csv",
    index=False
)

test_ratings.to_csv(
    "../datasets/processed/test_ratings.csv",
    index=False
)

print("Processed datasets saved successfully.")